# Document, Barcode, and QR Vision

> **Intermediate · Applied vision**


## Why this matters

Documents and codes reward careful preprocessing: perspective correction, denoising, binarization, and confidence-aware decoding.

**Where it appears:** Receipt extraction, scan cleanup, QR workflows, inventory systems, and form processing.


## Learning Objectives

- Pre-process images specifically to improve OCR accuracy (binarization, deskew, denoise)
- Run text recognition with pytesseract and parse structured output
- Build an OCR pipeline that reports per-word confidence, not just raw text
- Detect and decode QR codes using OpenCV's built-in QRCodeDetector
- Detect and decode 1D barcodes using pyzbar
- Handle multi-code images and decoding failures gracefully


## Prerequisites

06 Drawing and Geometric Transformations; 09 Thresholding and Morphology

Work through the examples in order. Change one parameter at a time, inspect the result, and record what changed.


## Core OpenCV APIs

deskewing, `pytesseract`, `cv2.QRCodeDetector`, barcode decoders

For every API below, identify its input type, important parameters, return value, and failure mode before reusing it.


## Conceptual Foundation


### OCR (Optical Character Recognition)

OCR accuracy depends heavily on pre-processing quality -- feeding a raw,
skewed, noisy photo into an OCR engine gives poor results even with a
good engine. The standard pre-processing chain is: grayscale -> denoise ->
binarize (Otsu or adaptive, from notebook 13) -> deskew (correct rotation).
This notebook uses `pytesseract` (a wrapper around the Tesseract OCR
engine) and focuses on the OpenCV-side pre-processing, which is usually
the actual lever available to improve results.


### Barcode and QR Code Detection

QR codes are 2D codes with built-in error correction (Reed-Solomon),
making them decodable even when partially damaged or at an angle --
OpenCV's `cv2.QRCodeDetector` handles perspective correction internally.
Traditional 1D barcodes (UPC, Code128) need a different decoder (this
notebook uses `pyzbar`, since OpenCV's own barcode module has less
widespread availability across builds). Both should be treated as
unreliable I/O: always check for and handle decode failures.


## Setup

Run this cell once. It finds the repository whether Jupyter was launched from
the project root or from `notebooks/`, then exposes the small shared helpers
used throughout the course.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLBACKEND", "Agg")

_candidates = (Path.cwd(), Path.cwd().parent)
REPO_ROOT = next(
    (path for path in _candidates if (path / "utils" / "cv_utils.py").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")

UTILS_DIR = REPO_ROOT / "utils"
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

import cv2
import matplotlib.pyplot as plt
import numpy as np

from cv_utils import Timer, ensure_dir, get_real_data, has_module, load_real_image, safe_imread, show, show_grid

print("OpenCV:", cv2.__version__)
print("Repository:", REPO_ROOT)


## Guided Lessons


## Part 1: OCR (Optical Character Recognition)


### 1. Building a synthetic 'scanned document'

Render text with OpenCV, then apply realistic degradation (slight rotation + noise) to have a genuine pre-processing problem to solve.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, ensure_dir, show_grid, has_module


clean_doc = load_real_image("images/documents", "text.png")
# If the image is large, resize it for display
h, w = clean_doc.shape[:2]
if w > 800:
    clean_doc = cv2.resize(clean_doc, (800, int(h * 800 / w)))

rot = cv2.getRotationMatrix2D(
    (w // 2, h // 2), 4, 1.0
)  # slight 4-degree skew, like a hand-scanned page
skewed_doc = cv2.warpAffine(clean_doc, rot, (w, h), borderValue=(255, 255, 255))
rng = np.random.default_rng(5)
degraded_doc = np.clip(
    skewed_doc.astype(np.int16) + rng.normal(0, 12, skewed_doc.shape), 0, 255
).astype(np.uint8)

show_grid([("clean", clean_doc), ("skewed + noisy (realistic scan)", degraded_doc)])

### 2. Pre-processing pipeline: denoise, binarize, deskew

Chain the fixes in the right order -- denoise before binarizing (otherwise noise becomes binary speckle), then deskew using the minimum-area-rectangle of text pixels.


In [ ]:
def deskew(binary: np.ndarray) -> np.ndarray:
    """Estimate and correct small rotation using the minAreaRect of foreground (text) pixels."""
    coords = np.column_stack(np.where(binary > 0))
    if len(coords) < 10:
        return binary
    angle = cv2.minAreaRect(coords)[-1]
    angle = -(90 + angle) if angle < -45 else -angle
    h, w = binary.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(binary, M, (w, h), borderValue=0)


def preprocess_for_ocr(image: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    denoised = cv2.fastNlMeansDenoising(gray, h=10)
    _, binary = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    straightened = deskew(binary)
    return cv2.bitwise_not(
        straightened
    )  # tesseract expects dark text on light background


ready_for_ocr = preprocess_for_ocr(degraded_doc)
show_grid(
    [
        ("degraded input", degraded_doc),
        ("preprocessed (denoised, binarized, deskewed)", ready_for_ocr),
    ]
)

### 3. Running OCR with per-word confidence

Use `pytesseract.image_to_data` (not just `image_to_string`) to get bounding boxes and per-word confidence -- essential for flagging low-confidence words for manual review instead of trusting every character blindly.


In [ ]:
def run_ocr_with_confidence(image: np.ndarray, min_confidence: int = 40) -> list[dict]:
    if not has_module("pytesseract"):
        print("pytesseract not installed.")
        return []
    import pytesseract

    try:
        data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT)
    except pytesseract.TesseractNotFoundError:
        print(
            "Tesseract binary not found! You must install Tesseract OCR on your system."
        )
        print(
            "Windows: Download from https://github.com/UB-Mannheim/tesseract/wiki and add to PATH."
        )
        print("Linux: sudo apt-get install tesseract-ocr")
        print("Mac: brew install tesseract")
        return []
    results = []
    for i, text in enumerate(data["text"]):
        conf = int(data["conf"][i])
        if text.strip() and conf >= min_confidence:
            results.append(
                {
                    "text": text,
                    "confidence": conf,
                    "box": (
                        data["left"][i],
                        data["top"][i],
                        data["width"][i],
                        data["height"][i],
                    ),
                }
            )
    return results


words = run_ocr_with_confidence(ready_for_ocr)
for w in words:
    print(w)

## Part 2: Barcode and QR Code Detection


### 1. Generating and detecting a QR code

Generate a real QR code with the `qrcode` library, then detect and decode it with OpenCV -- a genuine round trip, not a mocked example.


In [ ]:
import cv2
import numpy as np
from cv_utils import load_real_image, get_real_data, show_grid, has_module, ensure_dir

# 1. Load a real QR code image
qr_image = load_real_image("images/barcodes", "qr.png")

detector = cv2.QRCodeDetector()
decoded_text, points, straight_qr = detector.detectAndDecode(qr_image)
print("Decoded text:", decoded_text if decoded_text else "(decode failed)")

annotated = qr_image.copy()
if points is not None:
    pts = points.astype(int).reshape(-1, 2)
    cv2.polylines(annotated, [pts], True, (0, 0, 255), 3)
show_grid([("Real QR Image", qr_image), ("detected + outlined", annotated)])

### 2. Robustness: detecting a QR code under perspective distortion

Warp the QR code with a perspective transform (simulating a photo taken at an angle) and confirm OpenCV's detector still recovers it correctly -- this is the practical reason QR codes are preferred over 1D barcodes for camera-based scanning.


In [ ]:
h, w = qr_image.shape[:2]
src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
dst = np.float32(
    [[20, 40], [w - 5, 0], [w - 30, h - 10], [0, h - 40]]
)  # simulate a tilt
M = cv2.getPerspectiveTransform(src, dst)
tilted_qr = cv2.warpPerspective(qr_image, M, (w, h), borderValue=(255, 255, 255))

decoded_tilted, tilted_pts, _ = detector.detectAndDecode(tilted_qr)
annotated_tilted = tilted_qr.copy()
if tilted_pts is not None:
    pts = tilted_pts.astype(int).reshape(-1, 2)
    cv2.polylines(annotated_tilted, [pts], True, (255, 0, 0), 3)
print(
    "Decoded from tilted QR:", decoded_tilted if decoded_tilted else "(decode failed)"
)
show_grid([("tilted QR", tilted_qr), ("detected tilted QR", annotated_tilted)])

### 3. 1D barcode decoding with pyzbar, and failure handling

`pyzbar` handles both 1D barcodes and QR codes, returning a list -- always handle the empty-list case (no codes found) rather than assuming at least one result.


In [ ]:
# 2. Detect 1D Barcode using pyzbar (more robust than OpenCV native for barcodes)
barcode_image = load_real_image("images/barcodes", "ean13.png")


def decode_barcode(image: np.ndarray) -> list[dict]:
    if not has_module("pyzbar"):
        print(
            "pyzbar not installed -- install with: pip install pyzbar (and libzbar0 system package)"
        )
        return []
    from pyzbar.pyzbar import decode

    results = decode(image)
    return [
        {
            "type": r.type,
            "data": r.data.decode("utf-8", errors="replace"),
            "rect": r.rect,
        }
        for r in results
    ]


codes = decode_barcode(barcode_image)
annotated_barcode = barcode_image.copy()

if codes:
    for c in codes:
        print(f"Decoded Barcode: {c['data']} (Type: {c['type']})")
        x, y, w, h = c["rect"]
        cv2.rectangle(annotated_barcode, (x, y), (x + w, y + h), (0, 255, 0), 3)
else:
    print("Barcode decoding failed.")

show_grid(
    [("Real Barcode Image", barcode_image), ("detected barcode", annotated_barcode)]
)

## Mini Projects

Complete one project unaided before reading any provided solution. Extend it with a parameter, dataset, or failure case of your own.


### Mini Project — OCR (Optical Character Recognition): Structure-Aware Receipt Text Parser

OCR outputs are often flat lists of coordinate boxes. To extract key-value data (such as item prices on a receipt), we write a layout-aware post-processor that groups words sharing the same baseline.


In [ ]:
# Mock OCR data dictionary: text, top, left values
ocr_data = {
    "text": ["MILK", "$3.50", "BREAD", "$2.20", "TOTAL", "$5.70"],
    "left": [20, 220, 20, 220, 20, 220],
    "top": [50, 52, 100, 98, 200, 201],  # vertical pixel lines
}


def parse_line_items(data: dict) -> list[tuple[str, str]]:
    line_items = []
    # Group items where vertical distance |top_1 - top_2| <= 5 pixels
    n = len(data["text"])
    visited = [False] * n

    for i in range(n):
        if visited[i]:
            continue
        text_i = data["text"][i]
        top_i = data["top"][i]
        left_i = data["left"][i]

        # Find match on same line
        for j in range(i + 1, n):
            if not visited[j] and abs(data["top"][j] - top_i) <= 5:
                # Associate key-value based on horizontal coordinates
                val_text = data["text"][j]
                line_items.append((text_i, val_text))
                visited[j] = True
                visited[i] = True
                break

    return line_items


parsed = parse_line_items(ocr_data)
print("Extracted item-price mapping list:")
for item, price in parsed:
    print(f"  {item:8s} -> {price}")

### Mini Project — Barcode and QR Code Detection: Real-time Barcode Tracking Database Matcher

When scanning inventory codes in real-time, matching decoded values against a local database prevents double-scanning. Here, we build a tracking logic wrapper that simulates database checks for barcodes.


In [ ]:
# Simulated database of product inventory mapping barcodes to items
inventory_db = {
    "9780201379624": {"item": "C++ Book", "price": 45.0},
    "4902430232230": {"item": "Soda Can", "price": 1.50},
}


class BarcodeScannerTracker:
    def __init__(self):
        self.seen_codes = set()
        self.scanned_items = []

    def scan_code(self, code_str: str) -> None:
        if not code_str:
            return

        if code_str in self.seen_codes:
            return  # Skip duplicate scanning

        self.seen_codes.add(code_str)
        if code_str in inventory_db:
            info = inventory_db[code_str]
            self.scanned_items.append(info)
            print(f"SCANNED: {info['item']} | Price: ${info['price']:.2f}")
        else:
            print(f"SCANNED UNKNOWN CODE: {code_str}")


tracker = BarcodeScannerTracker()
# Feed the actually detected barcode into the tracker!
if codes:
    tracker.scan_code(codes[0]["data"])
    tracker.scan_code(codes[0]["data"])  # Duplicate check
tracker.scan_code("4902430232230")

## Exercises

Attempt the beginner, intermediate, and advanced prompts in order. Keep notes on assumptions and failures, not just successful output.


### Exercises — OCR (Optical Character Recognition)
1. Add a `--psm` (page segmentation mode) parameter to `run_ocr_with_confidence` and test PSM 6 vs PSM 11 on the document.
2. Measure OCR word count and mean confidence WITH vs WITHOUT the deskew step -- quantify its impact.
3. Extend the pipeline to draw bounding boxes around low-confidence words for manual review.

Use the empty cell below to work through them.


#### Solutions — OCR (Optical Character Recognition)

In [ ]:
# Solution 1: PSM parameter configuration
# In Tesseract, Page Segmentation Mode (PSM) specifies layout assumptions.
# PSM 6 assumes a single uniform block of text, which is ideal for standard paragraphs.
# PSM 11 runs sparse text localization, attempting to detect text fragments at arbitrary
# locations without structural grouping.


In [ ]:
# Solution 2: Word count and mean confidence impact of deskew step
# Deskewing aligns text horizontally. Standard OCR engines fail to segment text lines correctly
# if the skew angle exceeds 5-10 degrees, causing character recognition failure. Deskewing
# restores correct reading order, resulting in an increased valid word count and improved
# mean recognition confidence scores.


In [ ]:
# Solution 3: Draw bounding boxes around low-confidence words
def draw_low_confidence_filters(
    image: np.ndarray, ocr_results: dict, min_confidence: float = 60.0
) -> np.ndarray:
    """Draw red boxes around words that fall below min_confidence threshold."""
    canvas = image.copy()
    n = len(ocr_results.get("text", []))
    for i in range(n):
        conf = float(ocr_results["conf"][i])
        if 0 < conf < min_confidence:
            x = ocr_results["left"][i]
            y = ocr_results["top"][i]
            w = ocr_results["width"][i]
            h = ocr_results["height"][i]
            cv2.rectangle(canvas, (x, y), (x + w, y + h), (0, 0, 255), 2)
    return canvas

### Exercises — Barcode and QR Code Detection
1. Generate a QR code containing structured data (e.g. JSON) and parse it back with `json.loads` after decoding.
2. Test QR detection robustness against increasing Gaussian blur -- find the blur level where decoding starts failing.
3. Write `decode_with_retry` that tries multiple pre-processing variants (original, sharpened, thresholded) until one succeeds.

Use the empty cell below to work through them.


#### Solutions — Barcode and QR Code Detection

In [ ]:
# Solution 1: JSON payload decoding from QR code
import json


def parse_qr_json(qr_payload: str) -> dict:
    """Decode and parse JSON payload stored in QR code."""
    try:
        return json.loads(qr_payload)
    except json.JSONDecodeError:
        return {"error": "Invalid JSON format"}


# Verify with the payload we read from qr.png
if decoded_text:
    print("Parsed JSON data:", parse_qr_json(decoded_text))

In [ ]:
# Solution 2: QR detection robustness vs Gaussian blur
# Explanation: Gaussian blur smooths out high-frequency edge transitions in QR codes.
# As blur kernel size increases from 3 to 15, the contrast at the corner finder patterns
# degrades, preventing the decoder from locating coordinates, until decoding fails completely.

# Let's visualize this effect!
blur_plots = []
for k in [3, 9, 15]:
    blurred = cv2.GaussianBlur(qr_image, (k, k), 0)
    text, _, _ = detector.detectAndDecode(blurred)
    title = f"Blur {k}x{k} " + ("(OK)" if text else "(Failed)")
    blur_plots.append((title, blurred))
show_grid(blur_plots)

In [ ]:
# Solution 3: decode_with_retry multi-preprocessing wrapper
def decode_with_retry(image: np.ndarray, decoder_fn) -> str:
    """Try multiple image preprocessing passes until decoding succeeds."""
    # Attempt 1: Raw image
    res = decoder_fn(image)
    if res:
        return res

    # Attempt 2: Sharpened image
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    sharpened = cv2.filter2D(image, -1, kernel)
    res = decoder_fn(sharpened)
    if res:
        return res

    # Attempt 3: Adaptive thresholded binarization
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
    )
    return decoder_fn(thresh)


# Verify the retry logic
def my_qr_decoder(img):
    text, _, _ = detector.detectAndDecode(img)
    return text


print("Decoded with retry:", decode_with_retry(qr_image, my_qr_decoder))
# Let's visualize the preprocessing steps used in the retry logic:
kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
sharpened = cv2.filter2D(qr_image, -1, kernel)
gray = cv2.cvtColor(qr_image, cv2.COLOR_BGR2GRAY)
thresh = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2
)
show_grid(
    [("Raw Image", qr_image), ("Sharpened", sharpened), ("Adaptive Thresh", thresh)]
)

## Summary

You can build a defensible preprocessing path for text or codes and report confidence or failure instead of inventing a result.

- **Best Practices:** Preserve the source crop, tune preprocessing using representative scans, validate decoded payloads, and treat OCR output as uncertain data.
- **Common Pitfalls:** Hard-coding one lighting condition, trusting an empty/low-confidence OCR result, and using a decoded payload without validation.